# Lesson 6, Part 4

Let's code damped oscillation *x''*=-*bx'-cx* and test its performance absent friction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def damped_oscillation(b_friction,c_tension,x0,v0=0,maxtime=100,dt=0.1):
    time = np.arange(0,maxtime+dt/2,dt)
    num = len(time)
    x = np.zeros(num)
    x[0] = curr_x = x0
    curr_v = v0
    curr_a = - b_friction * curr_v - c_tension * curr_x
    for i in range(1,num):
        curr_v += curr_a * dt
        curr_x += curr_v * dt
        x[i] = curr_x 
        curr_a = - b_friction * curr_v - c_tension * curr_x
    return time,x

b_friction,c_tension,x0,v0 = 0,0.09,10,2
time,x = damped_oscillation(b_friction,c_tension,x0,v0)
plt.style.use('seaborn-v0_8')
plt.title('Undamped Oscillation')
plt.xlabel('time')
plt.ylabel('x')
plt.plot(time,x)
plt.show()

Let's check the stability of peaks over long time periods absent friction, 
and how sensitive the results are to the choice of *dt*.

In [ ]:
b_friction,c_tension,x0,v0 = 0,0.09,10,2
maxtime = 1000
for dt in (0.5,0.1,0.02):
    time,x = damped_oscillation(b_friction,c_tension,x0,v0,maxtime,dt)
    early_peak = np.max(x[:round(50/dt)])
    late_peak = np.max(x[-round(50/dt):])
    rel_diverge = np.log(late_peak/early_peak)
    print('Early peak','{:.4f}'.format(early_peak),'and late peak','{:.4f}'.format(late_peak),'given maxtime',maxtime,'and dt',dt)
    print('Relative change from early peak to late peak is','{:.1e}'.format(rel_diverge))

Hence *dt* = 0.1 seems a reasonable choice.
However, we can do better by applying trapezoidal approximations.

In [ ]:
def damped_oscillation_better(b_friction,c_tension,x0,v0,maxtime=100,dt=0.1):
    time = np.arange(0,maxtime+dt/2,dt)
    num = len(time)
    x = np.zeros(num)
    x[0] = curr_x = x0
    curr_v = v0
    curr_a = - b_friction * curr_v - c_tension * curr_x
    for i in range(1,num):
        prev_v = curr_v
        approx_v = prev_v + curr_a * dt
        approx_x = curr_x + 0.5 * (prev_v + approx_v) * dt
        prev_a = curr_a
        approx_a = - b_friction * approx_v - c_tension * approx_x
        curr_v += 0.5 * (prev_a + approx_a) * dt
        curr_x += 0.5 * (prev_v + curr_v) * dt
        x[i] = curr_x
        curr_a = - b_friction * curr_v - c_tension * curr_x
    return time,x

b_friction,c_tension,x0,v0 = 0,0.09,10,2
maxtime = 1000
print('Results with better approximation:')
for dt in (0.5,0.1,0.02):
    time,x = damped_oscillation_better(b_friction,c_tension,x0,v0,maxtime,dt)
    early_peak = np.max(x[:round(50/dt)])
    late_peak = np.max(x[-round(50/dt):])
    rel_diverge = np.log(late_peak/early_peak)
    print('Early peak','{:.4f}'.format(early_peak),'and late peak','{:.4f}'.format(late_peak),'given maxtime',maxtime,'and dt',dt)
    print('Relative change from early peak to late peak is','{:.1e}'.format(rel_diverge))

Let's check the match with the formulas derived in the text. 
The code here uses the numpy functions **sin** for sine and **cos** for cosine.

In [ ]:
def damped_oscillation_exact(b_friction,c_tension,x0,v0,maxtime=100,dt=0.1):
    time = np.arange(0, maxtime+dt/2, dt)
    num = len(time)
    v_mod = v0 + 0.5 * b_friction * x0   # modified v0
    s = (0.5 * b_friction)**2 - c_tension
    q = np.sqrt(abs(s))
    if s>0:
        w1 = 0.5 * (x0 + v_mod / q)
        w2 = 0.5 * (x0 - v_mod / q)
        y = w1 * np.exp(q * time)  + w2 * np.exp(-q * time)
    elif s==0:
        y = x0 + v_mod * time
    else:
        y = x0 * np.cos(q * time) + v_mod/q * np.sin(q * time)
    x = y * np.exp(-0.5 * b_friction * time)
    return time,x

print("Maximum divergence between simulations and exact formula:")
print(" b    original  better")
c_tension,x0,v0 = 0.09,10,2
for b_friction in (0,0.05,0.1,0.2,0.4,0.6,0.8): 
    time,x = damped_oscillation(b_friction,c_tension,x0,v0)
    time,xbetter = damped_oscillation_better(b_friction,c_tension,x0,v0)
    time,x_exact = damped_oscillation_exact(b_friction,c_tension,x0,v0)
    maxdiv =  np.max(np.abs(x - x_exact))
    maxdivbetter = np.max(np.abs(xbetter - x_exact))
    print('{:4.2f}'.format(b_friction),'{:7.3f}'.format(maxdiv),'{:8.3f}'.format(maxdivbetter))
plt.style.use('seaborn-v0_8')
plt.title('Compare paths for x')
plt.plot(time,x,time,xbetter,time,x_exact)
plt.show()

Let's plot damped oscillations for various levels of friction.

In [ ]:
c_tension,x0,v0 = 0.09,10,2
for b_friction in (0.8,0.1,0.2,0.4,0.6,0.8): 
    time,x = damped_oscillation_exact(b_friction,c_tension,x0,v0)
    b = round(b_friction,1)
    plt.plot(time,x,label='b = '+str(b))
plt.title('Damped Oscillation')
plt.legend()
plt.xlabel('time')
plt.ylabel('x')
plt.show()

When oscillations are overdamped, the spring will cross the origin at most once, 
and only if pushed hard enough initially toward the origin.

In [ ]:
b_friction,c_tension,x0 = 0.8,0.09,10
plt.title("Overdamped Oscillation")
for v0 in (4,0,-4,-8,-12):
    time,x = damped_oscillation_exact(b_friction,c_tension,x0,v0,maxtime=40)
    plt.plot(time,x,label='v0 = '+str(v0))
plt.legend()
plt.show()

When oscillations are underdamped, the initial extension and velocity 
affects the phases and amplitudes of cycles but not their periods.

In [ ]:
b_friction,c_tension,v0 = 0.05,0.09,5
plt.title("Underdamped Oscillations with different initial extensions")
for x0 in np.linspace(12,-12,5):
    time,x = damped_oscillation_exact(b_friction,c_tension,x0,v0,maxtime=100)
    plt.plot(time,x,label='x0 = '+str(x0))
plt.xlabel('time')
plt.legend()
x0 = 10
plt.figure()
plt.title("Underdamped Oscillations with different initial velocities")
for v0 in np.linspace(4,-12,5):
    time,x = damped_oscillation_exact(b_friction,c_tension,x0,v0,maxtime=100)
    plt.plot(time,x,label='v0 = '+str(v0))
    plt.ylabel('x')
plt.xlabel('time')
plt.legend()
plt.show()

Let's turn to logistic growth with *x'=-bx*(1-*x*). 
Here is a first pass at modeling it directly along 
with comparison to the exact formula from the text.

In [ ]:
def logistic_growth(b,x0,maxtime,num=1000):
    time = np.linspace(0,maxtime,num)
    dt = maxtime / (num - 1)
    x = np.zeros(num)
    x[0] = curr_x = x0
    for i in range(1,num):
        approx_x = curr_x + b * curr_x * (1 - curr_x) * dt 
        avg_x = 0.5 * (curr_x + approx_x)
        curr_x += b * avg_x * (1 - avg_x) * dt
        x[i] = curr_x
    # exact calculation
    coef = (1 - x0) / x0
    x_exact = 1 / (1 + coef * np.exp(-b * time))
    return time,x,x_exact

x0,maxtime = 0.01,30
print('Maximum relative divergence between simulation and exact formula')
for b in (0.2,0.5,2):
    time,x,x_exact = logistic_growth(b,x0,maxtime)
    plt.plot(time,x,label='b = '+str(b))
    plt.ylabel('x')
    rel_diverge = np.abs(np.log(x/x_exact))
    print('b =','{:.1f}'.format(b),':','{:.1e}'.format(np.max(rel_diverge)))
plt.title('Logistic Growth') 
plt.xlabel('time')
plt.legend()
plt.show()

While the empirical fits are ok, the chosen timescale is too short for *b* = 0.2 and too long for *b* = 2. 
The code below adjusts the timescale more appropriately and simplifies some computations.

In [ ]:
def logistic_growth_better(b, x0, xstop=0.995, num=1000):
    time = np.linspace(0,15/b,num)
    dt = time[-1] / (num - 1)
    btimesdt = b * dt
    x = np.zeros(num)
    x[0] = curr_x = x0
    for i in range(1,num):
        avg_x = curr_x + 0.5 * curr_x * (1 - curr_x) * btimesdt
        curr_x += avg_x * (1 - avg_x) * btimesdt
        x[i] = curr_x
        if curr_x > xstop:
            break
    # exact calculation
    coef = (1 - x0) / x0
    x_exact = 1 / (1 + coef * np.exp(-b * time[:i]))
    return time[:i],x[:i],x_exact

b,x0,maxtime = (0.2,2),0.01,30
print('Maximum relative divergence between simulation and exact formula')
for i in range(len(b)): 
    plt.subplot(len(b),1,i+1)
    plt.title('Logistic Growth Rescaled by ' + str(b[i]))
    time,x,x_exact = logistic_growth_better(b[i],x0)
    plt.plot(time,x,label='b ='+str(b[i]))
    plt.ylabel('x')
    plt.legend()
    plt.subplots_adjust(hspace=0.3)   # inserts some horizontal space between subplots
    rel_diverge = np.abs(np.log(x/x_exact))
    print('b =','{:.1f}'.format(b[i]),':','{:.1e}'.format(np.max(rel_diverge)))
plt.xlabel('time')
plt.show()

Here is a first pass at simulating a predator-prey model with *x'* = *Ax - Bxy* and *y'* = *Cxy - Dy*. 
Unfortunately, it does not induce the regular cycles described in the text.

In [ ]:
def predator_prey(A,B,C,D,x0,y0,maxtime=40,dt=0.01):
    time = np.arange(0,maxtime+dt/2,dt)
    num = len(time)
    x = np.zeros(num)
    y = np.zeros(num)
    x[0] = curr_x = x0
    y[0] = curr_y = y0
    for i in range(num):
        dx = (A - B * curr_y) * curr_x * dt
        dy = (C * curr_x - D) * curr_y * dt
        x[i] = curr_x = curr_x + dx
        y[i] = curr_y = curr_y + dy
    return time,x,y
        
A,B,C,D = 2,0.4,0.2,2
x0,y0 = 40,16
time,x,y = predator_prey(A,B,C,D,x0,y0)
plt.title('Predators and Prey (First Pass)')
plt.plot(time,x,time,y)
plt.show()

As usual, the model behaves much better when we use estimated average values over an interval.

In [ ]:
def predator_prey_better(A,B,C,D,x0,y0,maxtime=40,dt=0.01):
    time = np.arange(0,maxtime+dt/2,dt)
    num = len(time)
    x = np.zeros(num)
    y = np.zeros(num)
    x[0] = curr_x = x0
    y[0] = curr_y = y0
    for i in range(num):
        dx = (A - B * curr_y) * curr_x * dt
        dy = (C * curr_x - D) * curr_y * dt
        avg_x = curr_x + 0.5 * dx
        avg_y = curr_y + 0.5 * dy
        dx_better = (A - B * avg_y) * avg_x * dt
        dy_better = (C * avg_x - D) * avg_y * dt
        x[i] = curr_x = curr_x + dx_better
        y[i] = curr_y = curr_y + dy_better
    return time,x,y
        
A,B,C,D = 2,0.4,0.2,2
x0,y0 = 40,16
time,xbetter,ybetter = predator_prey_better(A,B,C,D,x0,y0)
plt.title('Predators and Prey (better)')
plt.plot(time,xbetter,time,ybetter)  
plt.show()

Ideally, the various (*x,y*) pairs maintain a constant value of *A* ln *y* - *By* - *Cx* + *D* ln *x*. 
While the first model fails badly in this respect, the second model comes very close.

In [ ]:
def predator_prey_orbit(A,B,C,D,x,y):
      return A * np.log(y) - B * y - C * x + D * np.log(x)
    
A,B,C,D = 2,0.4,0.2,2
x0,y0 = 40,16
time,x,y = predator_prey(A,B,C,D,x0,y0)
orbit_val = predator_prey_orbit(A,B,C,D,x,y)
time,xbetter,ybetter = predator_prey_better(A,B,C,D,x0,y0)
orbit_valbetter = predator_prey_orbit(A,B,C,D,xbetter,ybetter)
plt.subplot(2,1,1)
plt.title('Log orbit values (Original)')
plt.plot(time, orbit_val)
plt.subplots_adjust(hspace=0.3)
plt.subplot(2,1,2)
plt.title('Log orbit values (Better)')
plt.plot(time, orbit_valbetter)
plt.show()

Here are close-ups of three predator-prey models. Predator cycles always trail prey cycles. 
The transitions between upswings, downswings and stable periods often look quite abrupt.

In [ ]:
A,B,C,D = 2,0.4,0.2,2
maxtime = 15
x0,y0 = 40,16
plt.subplot(3,1,1)
plt.title('Examples of Fluctuations in Predators and Prey')
xy0 = [[40,16], [20,8], [10,4]]
for i in range(3): 
    plt.subplot(3,1,i+1)
    time,x,y = predator_prey_better(A,B,C,D,xy0[i][0],xy0[i][0],maxtime)
    plt.plot(time,x,time,y)
    plt.legend(['Prey','Predators'])
plt.show()